In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv('Backend\cleaneddata.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'cleaneddata.csv'

In [3]:
# Shape of the data
print('The shape of the recipe dataset is:', df.shape)

The shape of the recipe dataset is: (13493, 5)


In [4]:
df.head()

,Unnamed: 0,S.N,Title,Instructions,Ingredients
0,0,0,Miso-Butter Roast Chicken With Acorn Squash Pa...,"Pat chicken dry with paper towels, season all ...","['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher..."
1,1,1,Crispy Salt and Pepper Potatoes,Preheat oven to 400°F and line a rimmed baking...,"['2 large egg whites', '1 pound new potatoes (..."
2,2,2,Thanksgiving Mac and Cheese,Place a rack in middle of oven; preheat to 400...,"['1 cup evaporated milk', '1 cup whole milk', ..."
3,3,3,Italian Sausage and Bread Stuffing,Preheat oven to 350°F with rack in middle. Gen...,"['1 (¾- to 1-pound) round Italian loaf, cut in..."
4,4,4,Newton's Law,Stir together brown sugar and hot water in a c...,"['1 teaspoon dark brown sugar', '1 teaspoon ho..."


In [5]:
df.drop('Unnamed: 0', axis=1, inplace=True)

## Data Cleaning

In [7]:
# Info about the recipe dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13493 entries, 0 to 13492
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   S.N           13493 non-null  int64 
 1   Title         13493 non-null  object
 2   Instructions  13493 non-null  object
 3   Ingredients   13493 non-null  object
dtypes: int64(1), object(3)
memory usage: 421.8+ KB


In [8]:
# Convert 'id' and 'contributor_id' columns to object data type
df['S.N'] = df['S.N'].astype(str)


### Missing Data

In [10]:
# Checking for missing values
df.isna().sum()

S.N             0
Title           0
Instructions    0
Ingredients     0
dtype: int64

### Duplicate Data

In [12]:
# Duplicated rows
print("duplicated rows in recipes dataset:", df.duplicated().sum())

duplicated rows in recipes dataset: 0


### Outliers

In [14]:
# Summary statistics of recipes dataset
df.describe()

,S.N,Title,Instructions,Ingredients
count,13493,13493,13493,13493
unique,13493,13302,13464,13471
top,0,Potato Latkes,Place ingredients in blender in the order list...,['']
freq,1,5,5,6


In [15]:
print('['']' in set(df['Instructions']))

False


In [16]:
df.to_pickle('receipe.pkl')

In [17]:
data=df

## Content Based Recommendation

In [19]:
# Load data
data =pd.read_pickle("receipe.pkl")
data.shape

(13493, 4)

In [20]:
# Sample 25% of the dataset
sampled_data = data.sample(frac=0.25, random_state=42)

# View sample
display(sampled_data.head())

# Print the shape of the sampled dataset
print("Shape of Sampled Dataset:", sampled_data.shape)

,S.N,Title,Instructions,Ingredients
3859,3859,Crunchy Crab Salad,"To make the dressing, put the ginger and citru...","['200g fresh crabmeat', '1 avocado, peeled, st..."
833,833,Salted Plum Lychee Panna Cotta,Because agar-agar is seaweed-based and fairly ...,"['1 cup water', '2 teaspoons salted plum powde..."
3288,3288,Fried Chicken Thighs with Cheesy Grits,"Combine buttermilk, cayenne, garlic powder, sa...","['1 1/2 cups buttermilk', '1 teaspoon cayenne ..."
4315,4316,Roasted Beer and Lime Cauliflower Tacos with C...,Cut the cabbage into the thinnest strips you c...,['1/2 head of green cabbage (about 1/2 pound)'...
7460,7461,Stained-Glass Ornaments,"1. In a large bowl, whisk together the flour, ...","['3 cups all-purpose flour', '1 teaspoon bakin..."


Shape of Sampled Dataset: (3373, 4)


In [21]:
# import libraries
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\xenon\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
# Create a function for tokenizer

stemmer = nltk.stem.PorterStemmer()
ENGLISH_STOP_WORDS = stopwords.words('english')

def recipe_tokenizer(sentence):
    # remove punctuation and set to lower case
    for punctuation_mark in string.punctuation:
        sentence = sentence.replace(punctuation_mark,'').lower()

    # split sentence into words
    listofwords = sentence.split(' ')
    listofstemmed_words = []

    # remove stopwords and any tokens that are just empty strings
    for word in listofwords:
        if (not word in ENGLISH_STOP_WORDS) and (word!=''):
            # Stem words
            stemmed_word = stemmer.stem(word)
            listofstemmed_words.append(stemmed_word)

    return listofstemmed_words

In [23]:
# Import libraries

import gensim
from gensim.models import Word2Vec
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
# Function to create Word2Vec embeddings from 'ingredients'
def word_embedding(sampled_data, column):
    # Tokenize the data
    tokenized_data = sampled_data[column].apply(recipe_tokenizer)

    # Train a Word2Vec model
    model = Word2Vec(tokenized_data, vector_size=100, window=5, min_count=1, workers=4)

    # Create word embeddings for each word
    embeddings = {word: model.wv[word] for word in model.wv.index_to_key}
    
    return embeddings

In [25]:
# Function for word embedding using Word2Vec
def word_embedding(sampled_data, column):
    # Tokenize the text data
    tokenized_data = sampled_data[column].apply(recipe_tokenizer)

    # Train a Word2Vec model
    model = Word2Vec(tokenized_data, vector_size=100, window=5, min_count=1, workers=4)

    # Create word embeddings for each word in the vocabulary
    embeddings = {word: model.wv[word] for word in model.wv.index_to_key}

    return embeddings

In [26]:
# Function to pre-compute and store the combined embeddings
def precompute_embeddings(sampled_data):
    # Step 1: Process 'ingredients' using word2vec and create word embeddings
    embeddings = word_embedding(sampled_data, 'Ingredients')

    # Step 2: Concatenate relevant columns (excluding 'ingredients')
    sampled_data['text_data'] = sampled_data[['Title','Instructions']].astype(str).agg(' '.join, axis=1)

    # Step 3: Preprocess the text data (example: lowercase conversion)
    sampled_data['text_data'] = sampled_data['text_data'].str.lower()

    # Step 4: Vectorize the text data (excluding 'ingredients') using TF-IDF
    vectorizer = TfidfVectorizer(min_df=5,
                                 tokenizer=recipe_tokenizer)
    vectorized_data = vectorizer.fit_transform(sampled_data['text_data'])

    # Step 5: Retrieve the word embeddings for 'ingredients'
    ingredient_embeddings = [np.mean([embeddings[word] for word in recipe_tokenizer(ingredients) if word in embeddings]
                                      or [np.zeros(100)], axis=0) for ingredients in sampled_data['Ingredients']]

    # Step 6: Combine the vectorized data and ingredient embeddings
    combined_embeddings = np.concatenate([vectorized_data.toarray(), np.array(ingredient_embeddings)], axis=1)
    
    # Step 7: Store combined embeddings in pkl file
    with open('combined_embeddings.pkl', 'wb') as f:
        pickle.dump(combined_embeddings, f)
    
    # Step 8: Store the trained TF-IDF vectorizer model in a separate pkl file
    with open('tfidf_vectorizer.pkl', 'wb') as f:
        pickle.dump(vectorizer, f)
    
    # Step 9: Done!  
    print("Text data and TF-IDF vectorizer model stored in pkl files!")
    


In [27]:
# store vectorized data and the trained TF-IDF vectorizer model from sampled data
precompute_embeddings(sampled_data)

Text data and TF-IDF vectorizer model stored in pkl files!


In [30]:
# Function to load the combined embeddings and TF-IDF vectorizer model
def load_embeddings_and_vectorizer():
    with open('combined_embeddings.pkl', 'rb') as f:
        combined_embeddings = pickle.load(f)
    with open('tfidf_vectorizer.pkl', 'rb') as f:
        vectorizer = pickle.load(f)
    return combined_embeddings, vectorizer

# Function for finding recipes
def find_similar_recipes(sampled_data, user_input, num_similar=5):
    try:
        combined_embeddings, vectorizer = load_embeddings_and_vectorizer()
    except FileNotFoundError:
        precompute_embeddings(sampled_data)
        combined_embeddings, vectorizer = load_embeddings_and_vectorizer()

    # Process user input
    # Create a DataFrame for user input
    user_data = pd.DataFrame({'text_data': [user_input]})
    user_data['text_data'] = user_data['text_data'].str.lower()

    # Vectorize the user input using the provided vectorizer
    user_vectorized_data = vectorizer.transform(user_data['text_data'])

    # Ensure the number of features in user_vectorized_data matches with combined_embeddings
    num_missing_features = combined_embeddings.shape[1] - user_vectorized_data.shape[1]
    if num_missing_features > 0:
        # Add zero columns to user_vectorized_data to match the number of features
        user_vectorized_data = np.pad(user_vectorized_data.toarray(), ((0, 0), (0, num_missing_features)))

    # Compute cosine similarity with user input
    cosine_sim_matrix = cosine_similarity(user_vectorized_data, combined_embeddings)

    # Retrieve similar recipe indices
    similar_recipes = cosine_sim_matrix[0].argsort()[::-1][:num_similar]

    # Get similar recipe names from food_df
    similar_recipe_names = sampled_data.iloc[similar_recipes]['Title'].tolist()

    return similar_recipe_names

In [32]:
# Test
find_similar_recipes(sampled_data, "garlic")

['Beef Tenderloin with Garlic Horseradish Cream',
 'Roasted-Garlic Herb Dip',
 'Smashed Potatoes with Roasted-Garlic Gravy',
 'Tomato Bread Pudding',
 'D.I.Y. Steak Sauce']

In [34]:
# Save (pickle) the sampled dataset to a .pkl file
with open('sampled_data.pkl', 'wb') as f:
        pickle.dump(sampled_data, f)